In [7]:
########################################################################
# Inclusão das Bibliotecas Necessárias
########################################################################
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
########################################################################
# Localizando o Diretório Base
########################################################################
%cd /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código


/content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código


In [ ]:
########################################################################f

In [10]:
"""
IMPLEMENTAÇÃO VDN - VERSÃO 6.0.0
Correções v6 (análise completa ep100-800 da v5):
  1. Curriculum baseado em performance (≥70% sucesso) em vez de episódio fixo
  2. Warm-up de epsilon ao entrar em nova fase (ε=0.5 por 100 eps)
  3. PER: alpha 0.5→0.3, beta_start 0.4→0.6 (menos overfitting)
  4. TRAIN_FREQ: 4→8 (gradientes mais estáveis)
  5. EPSILON_END: 0.05→0.10 (nunca vai abaixo de 10% na exploração)
  6. Demais parâmetros mantidos da v5 (anti-loop, recompensas OK)
"""

import numpy as np
import gymnasium as gym
from gymnasium import spaces
import matplotlib.pyplot as plt
from collections import deque
import random
from typing import List, Optional
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from datetime import datetime
from pathlib import Path
import warnings
import time
import gc

warnings.filterwarnings('ignore')

# ==================== MAPA ====================
MAP_CONFIG = {
    'height': 12,
    'width': 8,
    'grid': [
        ['R1', '0', '0', '0', '0', '0', '0', 'R2'],
        ['0',  'Y', 'A', 'A', 'A', 'A', '0', '0'],
        ['0',  '0', 'A', 'A', 'A', 'A', '0', '0'],
        ['X',  '0', '0', '0', 'X', '0', 'Y', '0'],
        ['0',  'Y', 'X', '0', '0', '0', '0', '0'],
        ['0',  '0', '0', 'X', '0', 'Y', 'X', 'X'],
        ['0',  'X', '0', 'Y', '0', 'X', '0', '0'],
        ['0',  '0', '0', 'X', '0', '0', '0', '0'],
        ['X',  '0', 'Y', '0', '0', '0', 'X', '0'],
        ['X',  '0', 'B', 'B', 'B', 'B', 'X', '0'],
        ['X',  '0', 'B', 'B', 'B', 'B', 'Y', '0'],
        ['0',  '0', '0', 'Y', '0', '0', '0', '0'],
    ]
}

# ==================== CONFIGURAÇÃO V6 ====================
class VDNConfig:
    # ── Exploração ─────────────────────────────────────────────
    EPSILON_START        = 1.0
    EPSILON_END          = 0.10    # FIX v6: era 0.05 → mantém 10% exploração mínima
    EPSILON_DECAY_STEPS  = 350000  # mantido da v4/v5

    # FIX v6: warm-up ao entrar em nova fase de curriculum
    WARMUP_EPSILON       = 0.50    # epsilon forçado durante warm-up
    WARMUP_EPISODES      = 100     # quantos eps de warm-up ao trocar de fase

    # ── Aprendizado ────────────────────────────────────────────
    LEARNING_RATE        = 0.0001
    BATCH_SIZE           = 128
    GAMMA                = 0.97
    TAU                  = 0.003
    WEIGHT_DECAY         = 1e-5

    # ── Rede Neural ────────────────────────────────────────────
    HIDDEN_DIM           = 256
    DROPOUT_RATE         = 0.05

    # ── Memória ────────────────────────────────────────────────
    BUFFER_SIZE          = 250000
    ALPHA                = 0.3     # FIX v6: era 0.5 → menos overfitting por prioridade
    BETA_START           = 0.6     # FIX v6: era 0.4 → correção de viés mais forte
    BETA_END             = 1.0

    # ── Treinamento ────────────────────────────────────────────
    MAX_GRAD_NORM        = 5.0
    LEARNING_STARTS      = 3000
    TRAIN_FREQ           = 8       # FIX v6: era 4 → gradientes mais estáveis
    USE_SOFT_UPDATE      = True
    MAX_STEPS            = 500
    EPISODES_TOTAL       = 3000

    # ── Scheduler ──────────────────────────────────────────────
    SCHEDULER_PATIENCE   = 200
    SCHEDULER_FACTOR     = 0.7
    SCHEDULER_MIN_LR     = 5e-5
    SCHEDULER_START_EP   = 800

    # ── Recompensas (mantidas da v5 — OK) ─────────────────────
    REWARD_DELIVERY      = 80.0
    REWARD_PICKUP        = 8.0
    REWARD_ALL_DONE      = 120.0
    PENALTY_COLLISION    = -0.3
    PENALTY_FAILURE      = -0.2
    PENALTY_STEP         = -0.008
    PENALTY_BAD_ACTION   = -0.1
    PENALTY_LOOP         = -0.05
    SHAPING_APPROACH     = 0.1
    SHAPING_LEAVE        = -0.03
    REWARD_NEAR_TARGET   = 0.5

    # ── Anti-loop (mantido da v5) ──────────────────────────────
    LOOP_WINDOW          = 8
    LOOP_THRESHOLD       = 0.5

    # ── Curriculum baseado em PERFORMANCE (FIX v6 principal) ──
    CURRICULUM           = True
    # Sequência de fases: (n_caixas, sucesso_mínimo_para_avançar)
    CURRICULUM_PHASES    = [
        (2, 0.70),   # 2 caixas → avança quando ≥70% sucesso (100 eps)
        (4, 0.65),   # 4 caixas → avança quando ≥65% sucesso
        (6, 0.60),   # 6 caixas → avança quando ≥60% sucesso
        (8, 1.00),   # 8 caixas → fase final, nunca avança
    ]
    CURRICULUM_EVAL_WINDOW = 100   # janela de episódios para avaliar threshold
    # Proteção: mínimo de episódios em cada fase antes de poder avançar
    CURRICULUM_MIN_EPS   = 150

    # ── Sistema ────────────────────────────────────────────────
    SAVE_INTERVAL        = 500
    CLEAN_MEMORY_EVERY   = 500
    FAILURE_PROBABILITY  = 0.10
    ALTERNATIVE_DIRECTIONS = True


# ==================== CURRICULUM MANAGER ====================
class CurriculumManager:
    """
    Gerencia o curriculum baseado em performance.
    Avança de fase somente quando sucesso médio ≥ threshold
    e pelo menos CURRICULUM_MIN_EPS episódios foram feitos na fase atual.
    """

    def __init__(self, config: VDNConfig):
        self.config         = config
        self.phase_idx      = 0
        self.phase_ep_count = 0          # eps na fase atual
        self.warmup_count   = 0          # eps de warm-up restantes
        self.in_warmup      = False
        self.phase_history  = []         # (ep_global, n_boxes) para log
        self.transitions    = []         # eps em que houve transição

    @property
    def current_boxes(self) -> int:
        return self.config.CURRICULUM_PHASES[self.phase_idx][0]

    @property
    def current_threshold(self) -> float:
        return self.config.CURRICULUM_PHASES[self.phase_idx][1]

    def step(self, ep_global: int, recent_success: deque) -> dict:
        """
        Chamado a cada episódio.
        Retorna dict com estado atual e epsilon override se em warm-up.
        """
        self.phase_ep_count += 1
        self.phase_history.append(self.current_boxes)

        info = {
            'boxes':          self.current_boxes,
            'phase_idx':      self.phase_idx,
            'phase_ep':       self.phase_ep_count,
            'in_warmup':      self.in_warmup,
            'epsilon_override': None,
        }

        # Gerencia warm-up
        if self.in_warmup:
            self.warmup_count -= 1
            info['epsilon_override'] = self.config.WARMUP_EPSILON
            if self.warmup_count <= 0:
                self.in_warmup = False
                print(f"\n  ✅ Warm-up concluído → fase {self.phase_idx+1} "
                      f"({self.current_boxes} caixas) em modo normal\n")
            return info

        # Verifica se pode avançar de fase
        if (self.phase_idx < len(self.config.CURRICULUM_PHASES) - 1 and
                self.phase_ep_count >= self.config.CURRICULUM_MIN_EPS and
                len(recent_success) >= self.config.CURRICULUM_EVAL_WINDOW):

            avg_success = np.mean(list(recent_success)[-self.config.CURRICULUM_EVAL_WINDOW:])

            if avg_success >= self.current_threshold:
                self.phase_idx      += 1
                self.phase_ep_count  = 0
                self.in_warmup       = True
                self.warmup_count    = self.config.WARMUP_EPISODES
                self.transitions.append(ep_global)
                info['epsilon_override'] = self.config.WARMUP_EPSILON
                info['in_warmup']        = True
                info['boxes']            = self.current_boxes
                print(f"\n{'='*60}")
                print(f"  🎓 CURRICULUM AVANÇOU! ep {ep_global+1}")
                print(f"     Sucesso médio: {avg_success*100:.1f}% ≥ {self.current_threshold*100:.0f}%")
                print(f"     Nova fase: {self.current_boxes} caixas")
                print(f"     Iniciando warm-up de {self.config.WARMUP_EPISODES} eps (ε={self.config.WARMUP_EPSILON})")
                print(f"{'='*60}\n")

        return info

    def get_status(self) -> str:
        phase_name = f"Fase {self.phase_idx+1}/{len(self.config.CURRICULUM_PHASES)}"
        warmup_str = f" [WARMUP {self.warmup_count}ep]" if self.in_warmup else ""
        return f"{phase_name}{warmup_str} ({self.current_boxes}cx)"


# ==================== PRIORITIZED REPLAY BUFFER ====================
class PrioritizedReplayBuffer:
    def __init__(self, capacity: int, alpha: float = 0.3):
        self.capacity      = capacity
        self.alpha         = alpha
        self.buffer        = []
        self.priorities    = np.zeros(capacity, dtype=np.float32)
        self.position      = 0
        self._max_priority = 1.0

    def push(self, state, actions, rewards, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.position] = (state, actions, rewards, next_state, done)
        self.priorities[self.position] = self._max_priority ** self.alpha
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size: int, beta: float = 0.6):
        size  = len(self.buffer)
        probs = self.priorities[:size]
        probs = probs / probs.sum()
        indices = np.random.choice(size, batch_size, replace=False, p=probs)
        weights = (size * probs[indices]) ** (-beta)
        weights /= weights.max()
        batch   = [self.buffer[i] for i in indices]
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states,      dtype=np.float32),
                np.array(actions,     dtype=np.int64),
                np.array(rewards,     dtype=np.float32),
                np.array(next_states, dtype=np.float32),
                np.array(dones,       dtype=np.float32),
                indices,
                weights.astype(np.float32))

    def update_priorities(self, indices, td_errors):
        for idx, err in zip(indices, td_errors):
            p = (abs(err) + 1e-6) ** self.alpha
            self.priorities[idx] = p
            if p > self._max_priority:
                self._max_priority = p

    def __len__(self):
        return len(self.buffer)


# ==================== REDE NEURAL (DUELING DQN) ====================
class AgentNet(nn.Module):
    def __init__(self, input_dim: int, output_dim: int,
                 hidden_dim: int = 256, dropout: float = 0.05):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, output_dim)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.constant_(m.bias, 0.0)
        nn.init.orthogonal_(self.value_stream[-1].weight,     gain=0.01)
        nn.init.orthogonal_(self.advantage_stream[-1].weight, gain=0.01)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        f = self.feature(x)
        V = self.value_stream(f)
        A = self.advantage_stream(f)
        return V + A - A.mean(dim=1, keepdim=True)


# ==================== VDN CONTROLLER ====================
class VDNController:
    def __init__(self, n_agents: int, state_dim: int,
                 action_dim: int, config: VDNConfig):
        self.n_agents   = n_agents
        self.action_dim = action_dim
        self.config     = config
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"  VDN Controller → {self.device}")

        self.policy_nets = nn.ModuleList([
            AgentNet(state_dim, action_dim, config.HIDDEN_DIM, config.DROPOUT_RATE)
            for _ in range(n_agents)
        ]).to(self.device)

        self.target_nets = nn.ModuleList([
            AgentNet(state_dim, action_dim, config.HIDDEN_DIM, config.DROPOUT_RATE)
            for _ in range(n_agents)
        ]).to(self.device)

        for i in range(n_agents):
            self.target_nets[i].load_state_dict(self.policy_nets[i].state_dict())
            self.target_nets[i].eval()

        self.optimizer = optim.Adam(
            self.policy_nets.parameters(),
            lr=config.LEARNING_RATE,
            weight_decay=config.WEIGHT_DECAY
        )
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode='max',
            factor=config.SCHEDULER_FACTOR,
            patience=config.SCHEDULER_PATIENCE,
            min_lr=config.SCHEDULER_MIN_LR,
        )
        self.memory      = PrioritizedReplayBuffer(config.BUFFER_SIZE, alpha=config.ALPHA)
        self.steps_done  = 0
        self.learn_steps = 0
        self.losses      = []

    def _beta(self) -> float:
        frac = min(1.0, self.steps_done /
                   (self.config.EPISODES_TOTAL * self.config.MAX_STEPS))
        return self.config.BETA_START + frac * (
            self.config.BETA_END - self.config.BETA_START)

    def get_epsilon(self) -> float:
        if self.steps_done >= self.config.EPSILON_DECAY_STEPS:
            return self.config.EPSILON_END
        t = self.steps_done / self.config.EPSILON_DECAY_STEPS
        return self.config.EPSILON_START + t * (
            self.config.EPSILON_END - self.config.EPSILON_START)

    def select_actions(self, obs: np.ndarray,
                       training: bool = True,
                       epsilon_override: Optional[float] = None) -> list:
        self.steps_done += 1
        if epsilon_override is not None:
            eps = epsilon_override
        else:
            eps = self.get_epsilon() if training else 0.0

        actions = []
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(self.device)
            for net in self.policy_nets:
                if training and random.random() < eps:
                    actions.append(random.randrange(self.action_dim))
                else:
                    actions.append(net(obs_t).argmax(dim=1).item())
        return actions

    def remember(self, state, actions, rewards, next_state, done):
        self.memory.push(state, actions, rewards, next_state, done)

    def optimize(self) -> float:
        if (len(self.memory) < self.config.BATCH_SIZE or
                self.steps_done < self.config.LEARNING_STARTS):
            return 0.0
        self.learn_steps += 1
        if self.learn_steps % self.config.TRAIN_FREQ != 0:
            return 0.0

        states, actions, rewards, next_states, dones, indices, weights = \
            self.memory.sample(self.config.BATCH_SIZE, beta=self._beta())

        S  = torch.FloatTensor(states).to(self.device)
        A  = torch.LongTensor(actions).to(self.device)
        R  = torch.FloatTensor(rewards).to(self.device)
        S_ = torch.FloatTensor(next_states).to(self.device)
        D  = torch.FloatTensor(dones).to(self.device)
        W  = torch.FloatTensor(weights).to(self.device)

        q_total = torch.zeros(self.config.BATCH_SIZE, device=self.device)
        for i, net in enumerate(self.policy_nets):
            q_a = net(S).gather(1, A[:, i].unsqueeze(1)).squeeze(1)
            q_total = q_total + q_a

        with torch.no_grad():
            q_next = torch.zeros(self.config.BATCH_SIZE, device=self.device)
            for pnet, tnet in zip(self.policy_nets, self.target_nets):
                next_a = pnet(S_).argmax(dim=1, keepdim=True)
                q_next = q_next + tnet(S_).gather(1, next_a).squeeze(1)
            y = R.sum(dim=1) + self.config.GAMMA * q_next * (1 - D)

        td_errors = (y - q_total).detach().cpu().numpy()
        loss = (W * (y - q_total).pow(2)).mean()

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(
            self.policy_nets.parameters(), self.config.MAX_GRAD_NORM)
        self.optimizer.step()
        self.memory.update_priorities(indices, td_errors)

        if self.config.USE_SOFT_UPDATE:
            tau = self.config.TAU
            for pnet, tnet in zip(self.policy_nets, self.target_nets):
                for tp, pp in zip(tnet.parameters(), pnet.parameters()):
                    tp.data.copy_(tau * pp.data + (1 - tau) * tp.data)

        lv = loss.item()
        self.losses.append(lv)
        return lv

    def scheduler_step(self, metric: float, episode: int = 0):
        if episode >= self.config.SCHEDULER_START_EP:
            self.scheduler.step(metric)

    def get_lr(self) -> float:
        return self.optimizer.param_groups[0]['lr']

    def save(self, save_dir: Path, episode: int):
        save_dir.mkdir(parents=True, exist_ok=True)
        for i, net in enumerate(self.policy_nets):
            torch.save(net.state_dict(),
                       save_dir / f"vdn_agent{i}_ep{episode}.pth")

    def save_final(self, save_dir: Path):
        save_dir.mkdir(parents=True, exist_ok=True)
        for i, net in enumerate(self.policy_nets):
            torch.save(net.state_dict(),
                       save_dir / f"vdn_agent{i}_final.pth")


# ==================== AMBIENTE WAREHOUSE V6 ====================
class WarehouseEnv(gym.Env):
    metadata = {'render.modes': ['rgb_array']}

    def __init__(self, config: VDNConfig = None, active_boxes: int = 2):
        super().__init__()
        self.config       = config or VDNConfig()
        self.height       = MAP_CONFIG['height']
        self.width        = MAP_CONFIG['width']
        self.base_grid    = [row[:] for row in MAP_CONFIG['grid']]
        self.grid         = [row[:] for row in self.base_grid]
        self.active_boxes = active_boxes

        self.y_positions = self._find_base('Y')
        self.all_a_pos   = self._find_base('A')
        self.all_b_pos   = self._find_base('B')
        self.num_robots  = 2
        self.num_targets = len(self.all_b_pos)

        obs_dim = (self.num_robots * 2) + (8 * 2) + (self.num_targets * 2) + 10
        self.observation_space = spaces.Box(
            low=-1, high=self.height + self.width,
            shape=(obs_dim,), dtype=np.float32)
        self.action_space = spaces.Tuple(
            [spaces.Discrete(6)] * self.num_robots)
        self._init_ep()

    def _find_base(self, sym):
        return [(i, j) for i in range(self.height) for j in range(self.width)
                if self.base_grid[i][j].startswith(sym)]

    def _find_grid(self, sym):
        return [(i, j) for i in range(self.height) for j in range(self.width)
                if self.grid[i][j].startswith(sym)]

    def _init_ep(self):
        self.robot_positions   = None
        self.box_positions     = None
        self.delivered_boxes   = None
        self.targets           = None
        self.carrying          = [False] * self.num_robots
        self.steps             = 0
        self.total_deliveries  = 0
        self.collisions        = 0
        self.failures          = [0, 0]
        self.distance_traveled = [0, 0]
        self.prev_dist_box     = [0.0] * self.num_robots
        self.prev_dist_tgt     = [0.0] * self.num_robots
        w = self.config.LOOP_WINDOW
        self.pos_history = [deque(maxlen=w) for _ in range(self.num_robots)]

    def set_active_boxes(self, n: int):
        self.active_boxes = n

    def _remove_random_y(self):
        self.grid = [row[:] for row in self.base_grid]
        for (i, j) in self.y_positions:
            if random.random() < 0.5:
                self.grid[i][j] = '0'

    def reset(self):
        self._remove_random_y()
        self._init_ep()
        self.robot_positions = self._find_grid('R')
        chosen = random.sample(self.all_a_pos, self.active_boxes)
        self.box_positions   = list(chosen)
        self.delivered_boxes = [False] * self.active_boxes
        self.targets         = list(self.all_b_pos)
        self.prev_dist_box   = [self._dist_box(r) for r in range(self.num_robots)]
        self.prev_dist_tgt   = [self._dist_tgt(r) for r in range(self.num_robots)]
        for rid in range(self.num_robots):
            self.pos_history[rid].append(self.robot_positions[rid])
        return self._get_obs(), self._get_info()

    def _dist_box(self, rid: int) -> float:
        rp    = self.robot_positions[rid]
        boxes = [bp for bid, bp in enumerate(self.box_positions)
                 if bp is not None and not self.delivered_boxes[bid]]
        if not boxes:
            return 0.0
        return float(min(abs(rp[0]-bp[0]) + abs(rp[1]-bp[1]) for bp in boxes))

    def _dist_tgt(self, rid: int) -> float:
        rp = self.robot_positions[rid]
        return float(min(abs(rp[0]-tp[0]) + abs(rp[1]-tp[1]) for tp in self.targets))

    def _is_looping(self, rid: int) -> bool:
        hist = self.pos_history[rid]
        if len(hist) < self.config.LOOP_WINDOW:
            return False
        return len(set(hist)) / len(hist) < self.config.LOOP_THRESHOLD

    def _is_valid(self, pos, robot_id=None) -> bool:
        i, j = pos
        if not (0 <= i < self.height and 0 <= j < self.width):
            return False
        if self.grid[i][j] in ('X', 'Y'):
            return False
        if robot_id is not None:
            for rid, rp in enumerate(self.robot_positions):
                if rid != robot_id and rp == (i, j):
                    return False
        return True

    def _alt_dir(self, action: int, rid: int):
        i, j = self.robot_positions[rid]
        alts  = [a for a in range(4) if a != action]
        random.shuffle(alts)
        for a in alts:
            ni = i + [-1, 1, 0, 0][a]
            nj = j + [0, 0, -1, 1][a]
            if self._is_valid((ni, nj), rid):
                return (ni, nj)
        return None

    def _move(self, rid: int, action: int) -> float:
        i, j = self.robot_positions[rid]
        ni   = i + [-1, 1, 0, 0][action]
        nj   = j + [0, 0, -1, 1][action]
        r    = self.config.PENALTY_STEP

        if random.random() < self.config.FAILURE_PROBABILITY:
            self.failures[rid] += 1
            r += self.config.PENALTY_FAILURE
            if self.config.ALTERNATIVE_DIRECTIONS:
                alt = self._alt_dir(action, rid)
                if alt:
                    self.grid[i][j] = '0'
                    self.grid[alt[0]][alt[1]] = f'R{rid+1}'
                    self.robot_positions[rid] = alt
                    self.distance_traveled[rid] += 1
                    self.pos_history[rid].append(alt)
            return r

        if self._is_valid((ni, nj), rid):
            self.grid[i][j] = '0'
            self.grid[ni][nj] = f'R{rid+1}'
            self.robot_positions[rid] = (ni, nj)
            self.distance_traveled[rid] += 1
        else:
            self.collisions += 1
            r += self.config.PENALTY_COLLISION
        self.pos_history[rid].append(self.robot_positions[rid])
        return r

    def _pickup(self, rid: int) -> float:
        if self.carrying[rid]:
            return self.config.PENALTY_BAD_ACTION
        rp = self.robot_positions[rid]
        for bid, bp in enumerate(self.box_positions):
            if not self.delivered_boxes[bid] and bp == rp:
                self.box_positions[bid] = None
                self.carrying[rid] = True
                return self.config.REWARD_PICKUP
        return self.config.PENALTY_BAD_ACTION

    def _drop(self, rid: int) -> float:
        if not self.carrying[rid]:
            return self.config.PENALTY_BAD_ACTION
        rp          = self.robot_positions[rid]
        carrying_id = next(
            (bid for bid, bp in enumerate(self.box_positions)
             if bp is None and not self.delivered_boxes[bid]), None)
        if carrying_id is None:
            return self.config.PENALTY_BAD_ACTION
        if rp in self.targets:
            self.delivered_boxes[carrying_id] = True
            self.total_deliveries += 1
            self.carrying[rid] = False
            return self.config.REWARD_DELIVERY
        d = self._dist_tgt(rid)
        return self.config.PENALTY_BAD_ACTION + self.config.REWARD_NEAR_TARGET / max(d, 1.0)

    def _shaped(self, rid: int, base: float) -> float:
        r = base
        if self._is_looping(rid):
            r += self.config.PENALTY_LOOP
        if not self.carrying[rid]:
            cur, prev = self._dist_box(rid), self.prev_dist_box[rid]
            if cur < prev and not self._is_looping(rid):
                r += self.config.SHAPING_APPROACH * (prev - cur)
            elif cur > prev:
                r += self.config.SHAPING_LEAVE * (cur - prev)
            self.prev_dist_box[rid] = cur
        else:
            cur, prev = self._dist_tgt(rid), self.prev_dist_tgt[rid]
            if cur < prev and not self._is_looping(rid):
                r += self.config.SHAPING_APPROACH * (prev - cur)
            self.prev_dist_tgt[rid] = cur
        if all(self.delivered_boxes):
            r += self.config.REWARD_ALL_DONE
        return r

    def _get_obs(self) -> np.ndarray:
        obs = []
        for rp in self.robot_positions:
            obs += [rp[0] / self.height, rp[1] / self.width]
        for i in range(8):
            if i < self.active_boxes:
                bp = self.box_positions[i]
                if bp is None or self.delivered_boxes[i]:
                    obs += [-1.0, -1.0]
                else:
                    obs += [bp[0] / self.height, bp[1] / self.width]
            else:
                obs += [-1.0, -1.0]
        for tp in self.targets:
            obs += [tp[0] / self.height, tp[1] / self.width]
        for rid in range(self.num_robots):
            db = self._dist_box(rid) / (self.height + self.width)
            dt = self._dist_tgt(rid) / (self.height + self.width)
            c  = 1.0 if self.carrying[rid] else 0.0
            obs += [db, dt, c]
        obs += [self.total_deliveries / max(self.active_boxes, 1)]
        return np.array(obs, dtype=np.float32)

    def _get_info(self) -> dict:
        return {
            'steps':             self.steps,
            'total_deliveries':  self.total_deliveries,
            'num_boxes':         self.active_boxes,
            'collisions':        self.collisions,
            'failures_r1':       self.failures[0],
            'failures_r2':       self.failures[1],
            'distance_traveled': self.distance_traveled.copy(),
            'remaining_boxes':   sum(1 for d in self.delivered_boxes if not d),
            'success_rate':      self.total_deliveries / self.active_boxes,
        }

    def step(self, actions):
        self.steps += 1
        rewards = []
        for rid, act in enumerate(actions):
            if   act < 4:  base = self._move(rid, act)
            elif act == 4: base = self._pickup(rid)
            else:          base = self._drop(rid)
            rewards.append(self._shaped(rid, base))
        terminated = all(self.delivered_boxes)
        truncated  = self.steps >= self.config.MAX_STEPS
        return self._get_obs(), rewards, terminated, truncated, self._get_info()

    def close(self):
        pass


# ==================== PLOTAGEM ====================
def _ma(data, w=100):
    if len(data) < w:
        return None, None
    return list(range(w, len(data)+1)), np.convolve(data, np.ones(w)/w, mode='valid')


def plot_individual_graphs(metrics: dict, save_dir: Path,
                           config: VDNConfig, transitions: list):
    print(f"\n📊 Gerando gráficos em: {save_dir}")
    n  = len(metrics['episode_rewards'])
    ep = list(range(1, n+1))

    # Cores por fase (baseado em histórico real de caixas)
    phase_colors = ['#fff3cd', '#d4edda', '#cce5ff', '#f8d7da']

    def shade_real_phases(ax):
        """Sombrea fases com base no histórico real de caixas."""
        boxes_hist = metrics.get('active_boxes', [])
        if not boxes_hist:
            return
        phase_boxes = [p[0] for p in config.CURRICULUM_PHASES]
        prev_box = boxes_hist[0]
        start    = 0
        for i, b in enumerate(boxes_hist):
            if b != prev_box or i == len(boxes_hist) - 1:
                end   = i
                cidx  = phase_boxes.index(prev_box) if prev_box in phase_boxes else 0
                label = f'{prev_box} caixas (ep {start+1}–{end})'
                ax.axvspan(start+1, end+1, alpha=0.12,
                           color=phase_colors[cidx % len(phase_colors)],
                           label=label)
                prev_box = b
                start    = i
        # marca transições
        for t in transitions:
            ax.axvline(x=t+1, color='purple', ls='--', lw=1.5, alpha=0.7)

    def save_fig(path, title, xlabel, ylabel):
        ax = plt.gca()
        ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
        handles, labels = ax.get_legend_handles_labels()
        if handles:
            # remove duplicatas de legenda
            seen = {}
            for h, l in zip(handles, labels):
                if l not in seen:
                    seen[l] = h
            ax.legend(seen.values(), seen.keys(), fontsize=8,
                      loc='upper left', ncol=2)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(path, dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ {path.name}")

    # 1. Recompensa
    fig, ax = plt.subplots(figsize=(14, 7))
    shade_real_phases(ax)
    ax.plot(ep, metrics['episode_rewards'], color='steelblue', alpha=0.2, lw=1)
    x, ma = _ma(metrics['episode_rewards'])
    if ma is not None:
        ax.plot(x, ma, 'r-', lw=2.5, label='MM 100 ep')
    ax.axhline(y=0, color='gray', ls='--', lw=1, alpha=0.5)
    save_fig(save_dir/'grafico_recompensa_vdn.png',
             'Recompensa — VDN v6 (3000 ep)', 'Episódio', 'Recompensa')

    # 2. Entregas
    fig, ax = plt.subplots(figsize=(14, 7))
    shade_real_phases(ax)
    ax.plot(ep, metrics['episode_deliveries'], color='seagreen', alpha=0.2, lw=1)
    x, ma = _ma(metrics['episode_deliveries'])
    if ma is not None:
        ax.plot(x, ma, 'r-', lw=2.5, label='MM 100 ep')
    ax.plot(ep, metrics['active_boxes'], color='orange',
            ls='--', lw=1.5, alpha=0.9, label='Meta (caixas ativas)')
    save_fig(save_dir/'grafico_entregas_vdn.png',
             'Entregas — VDN v6 (3000 ep)', 'Episódio', 'Entregas')

    # 3. Steps
    fig, ax = plt.subplots(figsize=(14, 7))
    shade_real_phases(ax)
    ax.plot(ep, metrics['episode_steps'], color='darkorange', alpha=0.2, lw=1)
    x, ma = _ma(metrics['episode_steps'])
    if ma is not None:
        ax.plot(x, ma, 'r-', lw=2.5, label='MM 100 ep')
    save_fig(save_dir/'grafico_steps_vdn.png',
             'Steps — VDN v6 (3000 ep)', 'Episódio', 'Steps')

    # 4. Taxa de Sucesso
    fig, ax = plt.subplots(figsize=(14, 7))
    shade_real_phases(ax)
    ax.plot(ep, metrics['success_rates'], color='mediumpurple', alpha=0.2, lw=1)
    x, ma = _ma(metrics['success_rates'])
    if ma is not None:
        ax.plot(x, ma, 'r-', lw=2.5, label='MM Taxa Sucesso')
    ax.plot(ep, metrics['full_success'], color='gold', alpha=0.25, lw=1,
            label='Sucesso Completo (binário)')
    x2, ma2 = _ma(metrics['full_success'])
    if ma2 is not None:
        ax.plot(x2, ma2, color='darkgoldenrod', lw=2.5, label='MM Sucesso Completo')
    ax.axhline(y=0.95, color='green', ls='--', lw=1.5, label='Meta 95%')
    ax.axhline(y=0.70, color='limegreen', ls=':', lw=1.2, label='Threshold avanço fase')
    ax.set_ylim(-0.02, 1.12)
    save_fig(save_dir/'grafico_taxa_sucesso_vdn.png',
             'Taxa de Sucesso — VDN v6 (3000 ep)', 'Episódio', 'Taxa de Sucesso')

    # 5. Colisões
    fig, ax = plt.subplots(figsize=(14, 7))
    shade_real_phases(ax)
    ax.plot(ep, metrics['collisions'], color='crimson', alpha=0.2, lw=1)
    x, ma = _ma(metrics['collisions'])
    if ma is not None:
        ax.plot(x, ma, color='darkred', lw=2.5, label='MM 100 ep')
    save_fig(save_dir/'grafico_colisoes_vdn.png',
             'Colisões — VDN v6 (3000 ep)', 'Episódio', 'Colisões')

    # 6. Falhas
    fig, ax = plt.subplots(figsize=(14, 7))
    shade_real_phases(ax)
    total_f = np.array(metrics['failures_r1']) + np.array(metrics['failures_r2'])
    ax.plot(ep, total_f.tolist(), color='saddlebrown', alpha=0.25, lw=1, label='Total')
    ax.plot(ep, metrics['failures_r1'], color='red',  alpha=0.2, lw=1, label='Robô 1')
    ax.plot(ep, metrics['failures_r2'], color='blue', alpha=0.2, lw=1, label='Robô 2')
    x, ma = _ma(total_f.tolist())
    if ma is not None:
        ax.plot(x, ma, 'k-', lw=2.5, label='MM Total')
    save_fig(save_dir/'grafico_falhas_vdn.png',
             'Falhas — VDN v6 (3000 ep)', 'Episódio', 'Falhas')

    # 7. Distância
    fig, ax = plt.subplots(figsize=(14, 7))
    shade_real_phases(ax)
    ax.plot(ep, metrics['distance_traveled'], color='teal', alpha=0.2, lw=1)
    x, ma = _ma(metrics['distance_traveled'])
    if ma is not None:
        ax.plot(x, ma, 'r-', lw=2.5, label='MM 100 ep')
    save_fig(save_dir/'grafico_distancia_vdn.png',
             'Distância — VDN v6 (3000 ep)', 'Episódio', 'Distância')

    # 8. Loss
    if metrics.get('losses'):
        ls = metrics['losses']
        fig, ax = plt.subplots(figsize=(14, 7))
        ax.plot(range(1, len(ls)+1), ls, color='slategray', alpha=0.2, lw=1)
        w2 = max(min(500, len(ls)//5), 1)
        x, ma = _ma(ls, w=w2)
        if ma is not None:
            ax.plot(x, ma, 'r-', lw=2.5, label=f'MM ({w2})')
        ax.set(title='Loss — VDN v6', xlabel='Step', ylabel='Loss')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(save_dir/'grafico_loss_vdn.png', dpi=150, bbox_inches='tight')
        plt.close()
        print(f"  ✅ grafico_loss_vdn.png")

    # 9. LR + Epsilon
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
    shade_real_phases(ax1)
    if metrics.get('lr_history'):
        ax1.plot(range(1, len(metrics['lr_history'])+1),
                 metrics['lr_history'], color='navy', lw=2, label='LR')
        ax1.axvline(x=config.SCHEDULER_START_EP, color='red',
                    ls='--', lw=1.5, label=f'Scheduler ep {config.SCHEDULER_START_EP}')
    ax1.set(ylabel='Learning Rate', title='LR e Epsilon — VDN v6 (3000 ep)')
    ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

    shade_real_phases(ax2)
    if metrics.get('epsilon_history'):
        ax2.plot(range(1, len(metrics['epsilon_history'])+1),
                 metrics['epsilon_history'], color='darkorange', lw=2, label='Epsilon')
        ax2.axhline(y=config.EPSILON_END, color='gray',
                    ls='--', lw=1, label=f'ε min ({config.EPSILON_END})')
        ax2.axhline(y=config.WARMUP_EPSILON, color='purple',
                    ls=':', lw=1, label=f'Warm-up ε ({config.WARMUP_EPSILON})')
    ax2.set(xlabel='Episódio', ylabel='Epsilon')
    ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir/'grafico_lr_epsilon_vdn.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ grafico_lr_epsilon_vdn.png")

    # 10. Progresso do curriculum
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.step(ep, metrics['active_boxes'], color='steelblue', lw=2, where='post',
            label='Caixas ativas')
    for t in transitions:
        ax.axvline(x=t+1, color='purple', ls='--', lw=1.5, alpha=0.8)
        ax.text(t+5, max(metrics['active_boxes'])-0.3, f'ep {t+1}',
                fontsize=8, color='purple')
    ax.set(xlabel='Episódio', ylabel='N° de Caixas',
           title='Progresso do Curriculum — VDN v6 (baseado em performance)',
           ylim=[0, 9])
    ax.set_yticks([2, 4, 6, 8])
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_dir/'grafico_curriculum_vdn.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✅ grafico_curriculum_vdn.png")

    print(f"📊 Todos os gráficos salvos em: {save_dir}")


def save_metrics_csv(metrics: dict, save_dir: Path):
    n = len(metrics['episode_rewards'])
    df = pd.DataFrame({
        'episodio':         range(1, n+1),
        'recompensa':       metrics['episode_rewards'],
        'entregas':         metrics['episode_deliveries'],
        'caixas_ativas':    metrics['active_boxes'],
        'phase_idx':        metrics.get('phase_idx',       [None]*n),
        'in_warmup':        metrics.get('in_warmup',        [None]*n),
        'steps':            metrics['episode_steps'],
        'taxa_sucesso':     metrics['success_rates'],
        'sucesso_completo': metrics['full_success'],
        'colisoes':         metrics['collisions'],
        'falha_r1':         metrics['failures_r1'],
        'falha_r2':         metrics['failures_r2'],
        'distancia':        metrics['distance_traveled'],
        'lr':               metrics.get('lr_history',      [None]*n),
        'epsilon':          metrics.get('epsilon_history',  [None]*n),
    })
    p = save_dir / 'metricas_treinamento_vdn.csv'
    df.to_csv(p, index=False)
    print(f"📊 CSV salvo: {p}")


# ==================== TREINAMENTO ====================
def run_vdn_training(num_episodes: int = 3000):
    config = VDNConfig()
    config.EPISODES_TOTAL = num_episodes

    print("=" * 80)
    print("🤖 VDN v6 — CURRICULUM BASEADO EM PERFORMANCE + WARM-UP")
    print("=" * 80)
    print(f"\n📋 CORREÇÕES v6 vs v5:")
    print(f"   ✔ Curriculum: episódio fixo → performance ≥70/65/60%")
    print(f"   ✔ Min {config.CURRICULUM_MIN_EPS} eps por fase antes de avançar")
    print(f"   ✔ Warm-up {config.WARMUP_EPISODES} eps (ε={config.WARMUP_EPSILON}) ao trocar de fase")
    print(f"   ✔ PER alpha: 0.5→0.3  beta_start: 0.4→0.6")
    print(f"   ✔ TRAIN_FREQ: 4→8")
    print(f"   ✔ EPSILON_END: 0.05→0.10")
    print(f"\n📐 FASES DO CURRICULUM:")
    for i, (boxes, thresh) in enumerate(config.CURRICULUM_PHASES):
        nxt = f"→ {config.CURRICULUM_PHASES[i+1][0]}cx" if i < len(config.CURRICULUM_PHASES)-1 else "→ fim"
        print(f"   Fase {i+1}: {boxes} caixas  |  avança quando sucesso ≥ {thresh*100:.0f}%  {nxt}")
    print("=" * 80)

    env_tmp = WarehouseEnv(config=config, active_boxes=8)
    obs, _  = env_tmp.reset()
    state_dim  = len(obs)
    action_dim = 6
    env_tmp.close()
    print(f"\n📊 state_dim={state_dim}  action_dim={action_dim}  agentes=2")

    controller  = VDNController(2, state_dim, action_dim, config)
    curriculum  = CurriculumManager(config)

    metrics = {k: [] for k in [
        'episode_rewards', 'episode_deliveries', 'active_boxes',
        'phase_idx', 'in_warmup',
        'episode_steps', 'success_rates', 'full_success',
        'collisions', 'failures_r1', 'failures_r2',
        'distance_traveled', 'losses', 'lr_history', 'epsilon_history']}

    timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
    results_dir = Path(f"vdn_results_v6_{timestamp}")
    results_dir.mkdir(exist_ok=True)
    print(f"📁 Resultados: {results_dir.absolute()}\n")

    env = WarehouseEnv(config=config, active_boxes=2)
    t0  = time.time()
    recent_success = deque(maxlen=config.CURRICULUM_EVAL_WINDOW)

    for ep in range(num_episodes):
        # Curriculum step — pode mudar n_boxes e retornar epsilon_override
        cur_info = curriculum.step(ep, recent_success)
        n_boxes  = cur_info['boxes']
        eps_ovr  = cur_info['epsilon_override']

        env.set_active_boxes(n_boxes)
        obs, _ = env.reset()
        ep_reward = 0.0

        for step in range(config.MAX_STEPS):
            actions = controller.select_actions(obs, training=True,
                                                epsilon_override=eps_ovr)
            nobs, rewards, term, trunc, info = env.step(actions)
            done = term or trunc
            controller.remember(obs, actions, rewards, nobs, float(done))
            lv = controller.optimize()
            if lv:
                metrics['losses'].append(lv)
            ep_reward += sum(rewards)
            obs = nobs
            if done:
                break

        full = 1.0 if info['total_deliveries'] == n_boxes else 0.0
        recent_success.append(full)
        controller.scheduler_step(np.mean(recent_success), episode=ep)

        metrics['episode_rewards'].append(ep_reward)
        metrics['episode_deliveries'].append(info['total_deliveries'])
        metrics['active_boxes'].append(n_boxes)
        metrics['phase_idx'].append(cur_info['phase_idx'])
        metrics['in_warmup'].append(int(cur_info['in_warmup']))
        metrics['episode_steps'].append(step + 1)
        metrics['success_rates'].append(info['success_rate'])
        metrics['full_success'].append(full)
        metrics['collisions'].append(info['collisions'])
        metrics['failures_r1'].append(info['failures_r1'])
        metrics['failures_r2'].append(info['failures_r2'])
        metrics['distance_traveled'].append(sum(info['distance_traveled']))
        metrics['lr_history'].append(controller.get_lr())
        metrics['epsilon_history'].append(
            eps_ovr if eps_ovr is not None else controller.get_epsilon())

        if (ep + 1) % 100 == 0:
            w   = 100
            rr  = metrics['episode_rewards'][-w:]
            rd  = metrics['episode_deliveries'][-w:]
            rfs = list(recent_success)
            eps = metrics['epsilon_history'][-1]
            lr  = controller.get_lr()
            el  = (time.time() - t0) / 60
            buf = len(controller.memory)
            status = curriculum.get_status()
            print(f"Ep {ep+1:4d}/{num_episodes} | {status} | "
                  f"Reward: {np.mean(rr):8.2f} | "
                  f"Entrega: {np.mean(rd):.2f}/{n_boxes} | "
                  f"Sucesso: {np.mean(rfs)*100:5.1f}% | "
                  f"ε: {eps:.3f} | lr: {lr:.2e} | "
                  f"buf: {buf} | {el:.1f}min")

        if (ep + 1) % config.SAVE_INTERVAL == 0:
            controller.save(results_dir / "models", ep + 1)

        if (ep + 1) % config.CLEAN_MEMORY_EVERY == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    total_time = (time.time() - t0) / 60
    env.close()

    print(f"\n💾 Salvando resultados...")
    controller.save_final(results_dir / "models")
    save_metrics_csv(metrics, results_dir)
    plot_individual_graphs(metrics, results_dir, config, curriculum.transitions)

    w  = 100
    fr = metrics['episode_rewards'][-w:]
    fd = metrics['episode_deliveries'][-w:]
    fs = metrics['full_success'][-w:]
    fc = metrics['collisions'][-w:]

    report = f"""
========================================
RELATÓRIO FINAL — VDN v6 (3000 EPISÓDIOS)
========================================
DATA:      {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
DIRETÓRIO: {results_dir.absolute()}

CORREÇÕES v6 vs v5:
  Curriculum:    episódio fixo → performance ≥ threshold
  Warm-up:       {config.WARMUP_EPISODES} eps (ε={config.WARMUP_EPSILON}) ao trocar de fase
  PER alpha:     0.5 → 0.3
  PER beta_start: 0.4 → 0.6
  TRAIN_FREQ:    4 → 8
  EPSILON_END:   0.05 → 0.10

FASES REALIZADAS:
  Transições nos eps: {curriculum.transitions}
  Fase final atingida: {curriculum.current_boxes} caixas

MÉTRICAS (últimos 100 episódios):
  Recompensa média:     {np.mean(fr):.2f}
  Entregas médias:      {np.mean(fd):.2f}/8
  Sucesso completo:     {np.mean(fs)*100:.1f}%
  Colisões médias:      {np.mean(fc):.1f}
  Tempo total:          {total_time:.1f} min
  LR final:             {metrics['lr_history'][-1]:.2e}

MELHORES RESULTADOS:
  Melhor recompensa:  {max(metrics['episode_rewards']):.2f}
  Máx entregas/ep:    {max(metrics['episode_deliveries'])}/8
  Menor steps:        {min(metrics['episode_steps'])}

GRÁFICOS (10):
  grafico_recompensa_vdn.png    grafico_entregas_vdn.png
  grafico_steps_vdn.png         grafico_taxa_sucesso_vdn.png
  grafico_colisoes_vdn.png      grafico_falhas_vdn.png
  grafico_distancia_vdn.png     grafico_loss_vdn.png
  grafico_lr_epsilon_vdn.png    grafico_curriculum_vdn.png
========================================
"""
    with open(results_dir / "relatorio_final_vdn.txt", 'w', encoding='utf-8') as f:
        f.write(report)
    print(report)
    return controller, metrics, results_dir, curriculum


# ==================== MAIN ====================
if __name__ == "__main__":
    print("\n" + "=" * 80)
    print("🎮 VDN v6 — INICIANDO")
    print("=" * 80)
    try:
        controller, metrics, results_dir, curriculum = \
            run_vdn_training(num_episodes=3000)
        w = 100
        print("\n" + "=" * 60)
        print("✨ VDN v6 CONCLUÍDO! ✨")
        print("=" * 60)
        print(f"  Recompensa média (ult.{w}): {np.mean(metrics['episode_rewards'][-w:]):.2f}")
        print(f"  Entregas médias  (ult.{w}): {np.mean(metrics['episode_deliveries'][-w:]):.2f}/8")
        print(f"  Sucesso completo (ult.{w}): {np.mean(metrics['full_success'][-w:])*100:.1f}%")
        print(f"  Colisões médias  (ult.{w}): {np.mean(metrics['collisions'][-w:]):.1f}")
        print(f"  Fases atingidas:            {curriculum.phase_idx + 1}/4")
        print(f"  Transições em eps:          {curriculum.transitions}")
    except Exception as e:
        import traceback
        print(f"\n❌ ERRO: {e}")
        traceback.print_exc()


🎮 VDN v6 — INICIANDO
🤖 VDN v6 — CURRICULUM BASEADO EM PERFORMANCE + WARM-UP

📋 CORREÇÕES v6 vs v5:
   ✔ Curriculum: episódio fixo → performance ≥70/65/60%
   ✔ Min 150 eps por fase antes de avançar
   ✔ Warm-up 100 eps (ε=0.5) ao trocar de fase
   ✔ PER alpha: 0.5→0.3  beta_start: 0.4→0.6
   ✔ TRAIN_FREQ: 4→8
   ✔ EPSILON_END: 0.05→0.10

📐 FASES DO CURRICULUM:
   Fase 1: 2 caixas  |  avança quando sucesso ≥ 70%  → 4cx
   Fase 2: 4 caixas  |  avança quando sucesso ≥ 65%  → 6cx
   Fase 3: 6 caixas  |  avança quando sucesso ≥ 60%  → 8cx
   Fase 4: 8 caixas  |  avança quando sucesso ≥ 100%  → fim

📊 state_dim=43  action_dim=6  agentes=2
  VDN Controller → cuda
📁 Resultados: /content/drive/MyDrive/Atividades/PUC-DI/LocalMultiAgente/Código/vdn_results_v6_20260615_213758

Ep  100/3000 | Fase 1/4 (2cx) | Reward:    15.68 | Entrega: 0.84/2 | Sucesso:  14.0% | ε: 0.876 | lr: 1.00e-04 | buf: 48228 | 1.1min
Ep  200/3000 | Fase 1/4 (2cx) | Reward:   110.06 | Entrega: 1.34/2 | Sucesso:  46.0% | ε:

In [ ]:
########################################################################